# Are the flagged contaminants really separate sources?

A sanity check on `diffracc.data.cutout_quality`. This notebook plots random LoTSS-DR2 cutouts with PyBDSF-derived Gaussian components overlayed on the image, colouring each component by the source it belongs to (the `Parent_Source` field).

* **white, solid** ellipses = the cutout's own source (the target it is centred on),
* **coloured, dashed** ellipses = foreign sources (a different `Parent_Source`) — the contaminants,

each labelled with its catalogued peak flux (mJy/beam). If the contaamination flag is right, the coloured ellipses likely sit on visibly distinct blobs, away from the white target(s) at the centre.

The drawing lives in `diffracc.plotting.cutout_overlays` (so the same figures are reproducible outside this notebook); this notebook is just a driver.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Make `import diffracc` work regardless of where the notebook is launched from.
_repo = Path.cwd()
while not (_repo / "diffracc").is_dir() and _repo != _repo.parent:
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

import numpy as np
import pandas as pd

from diffracc.plotting.cutout_overlays import CutoutOverlayPlotter
from diffracc.utils import paths

RNG = np.random.default_rng(42)     # change the seed to draw a different random sample
print("repo root:", _repo)

## Load the flags and the plotter

`CutoutOverlayPlotter` loads the component and source catalogues once (takes a moment). The flags are the CSV written
by `python -m diffracc.data.cutout_quality`, but it would also be easy to compute them live using `compute_from_catalogues`.

In [ ]:
flags = pd.read_csv(paths.PREPROCESSING_PARENT / "cutout_quality_flags.csv")
plotter = CutoutOverlayPlotter(flags=flags)

foreign = flags["foreign_contaminant"].to_numpy().astype(bool)
n_det = flags["n_foreign_detected"].to_numpy()
cropped = flags["cropped"].to_numpy().astype(bool)
shares_host = flags["foreign_shares_optical_id"].to_numpy().astype(bool)
assert len(flags) == len(plotter.src_ra), "flags are not aligned with the resolved catalogue"
print(f"{foreign.sum():,} / {len(foreign):,} resolved cutouts flagged foreign-contaminated "
      f"({foreign.mean()*100:.1f}%); of those {shares_host.sum():,} share the target's optical ID "
      f"(soft possible-mis-split flag).")


def sample_existing(pool: np.ndarray, k: int) -> list[int]:
    """
    Sample k cutouts from the given pool of indices, but only return those that actually exist on disk.
    
    Parameters
    ----------
    pool : np.ndarray
        Array of indices to sample from.
    k : int
        Number of cutouts to sample.
    
    Returns
    -------
    list[int]
        List of indices of cutouts that exist on disk, sampled from the pool.
    """
    out = []
    for i in RNG.permutation(pool):
        if plotter.cutout_path(int(i)).exists():
            out.append(int(i))
        if len(out) == k:
            break
    return out

## Flagged (foreign-contaminated) cutouts

A cutout is flagged as contaminated if any part of a >=5 sigma component belonging to a different `Source_Parent` is inside the cutout boundary. Below we can plot cutouts flagged this way and any relevant components on them, using white for the target and colouring anything foreign.

You may need to edit the RNG seed above to see all of these, but you should notice some of these cares:

* **bright target source, dim contaminants** - the 'best' case for a contaminated cutout. Sometimes the contaminants are so dim (<5 sigma) that they're likely going to be considered as part of the noise background; otherwise, the cutout image largely reflects the target/catalogue source.
* **similar brightness sources and contaminants** - while these cutouts will read physical values such as peak flux correctly, and therefore be appropriate for the diffusion model's guidance, they do teach unmotivated structure/extent due to multiple emission components being present in an image intending to represently solely the target source.
* **dim target source, bright contaminants** - a very poor case, as this fails in multiple regards. It teaches guiding very poorly, as now an image for a e.g., 1 mJy source instead contains pixel values representative of an e.g., 30 mJy source; and, as cutouts are centred on the target source, the dominant emission in the image would be off-centre. These images will likely degrade model prediction quality.
* **bright, cut-off contaminants** - in these images, the plot is widened to show the full extent of every component - in reality, the cutout has a hard boundary, and this can cut-off contaminant ellipses, leading to a bright non-Gaussian 'blob' on the edge. Diffusion models are adept at capturing features like these, so if such cutouts are common in the dataset, this artifact can be replicated at inference time.

In [ ]:
picks = sample_existing(np.where(foreign & (n_det >= 1))[0], 12)
plotter.plot_grid(picks, suptitle="Foreign-contaminated cutouts  |  white = target,  colours = separate catalogued sources");

We can also view some information about each image:

* **index** - the index of the image in the dataset, which is ordered by RA. This also corresponds to the cutout number e.g., 'cutout1234.fits'.
* **source** - the name of the target source (`Source_Name`) in catalogue.
* **target_source_mJy** - the peak flux of the target source in mJy.
* **n_foreign(>=5sigma)** - the number of foreign components over 5 sigma that are detected in the cutout.
* **brightest_foreign_mJy** - the peak flux of the brightest contaminant in mJy. If this value is significantly greater than `target_source_mJy`, the cutout is a particularly poor image of the target source.
* **brightest_foreign_SNR** - the SNR of the brightest contaminant.
* **shares_optical_id** - whether the identified brightest contaminant shares the same optical ID (`ID_NAME`) as the target source. `Source_Parent` is currently used to link components to sources, but those are radio-based (e.g., PyBDSF or visual inspection); the Hardcastle et al. (2023) paper also identifies if LoTSS-DR2 sources belong to the same optical source. `ID_NAME` and `Source_Parent` do not completely agree with each other, but are determined in the same pipeline; it is only shown here for record purposes and not used in the program.

In [ ]:
# The same sources tabulated, including the soft optical mis-split flag.
pd.DataFrame({
    "index": picks,
    "source": [plotter.src_name[i] for i in picks],
    "target_flux_mJy": [round(float(flags["peak_flux_mjy"][i]), 3) for i in picks],
    "n_foreign(>=5sigma)": [int(n_det[i]) for i in picks],
    "brightest_foreign_mJy": [round(float(flags["brightest_foreign_flux"][i]), 3) for i in picks],
    "brightest_foreign_SNR": [round(float(flags["brightest_foreign_snr"][i]), 1) for i in picks],
    "shares_optical_id": [bool(shares_host[i]) for i in picks],
}).set_index("index")

## Cropped cutouts &mdash; is the target itself running off the edge?

Separate from foreign contamination, `cropped` asks whether the target's own emission extends past the 120&Prime; cutout frame: `flag_cropped_sources` walks the target's own components and tests, from each fitted `Maj`/`Min`/`PA` ellipse, whether any part crosses the frame boundary. Preprocessing already filters for sources with Largest Angular Size (LAS) greater than 120", but it is possible in the cases of e.g., multi-component sources, where the centre of the cutout is the flux-weighted centroid, for components to extend past the boundary and be `cropped`.

A cropped source is one whose morphology we would only partly capture in the cutout image. While the size of the cutout images is arbitrary, and in a real detector mosaic such cropped sources should cause no issue, we have to present the diffusion model with a set of same-sized images, and it is preferable to show a source's full extent rather than a cropped image.

In [ ]:
print(f"{cropped.sum():,} / {len(cropped):,} resolved cutouts flagged cropped ({cropped.mean()*100:.1f}%).")
crop_picks = sample_existing(np.where(cropped)[0], 12)
plotter.plot_grid(crop_picks, suptitle="Cropped cutouts  |  the target's own emission (white) crosses the frame edge");

## Clean cutouts, for contrast

We can also show sources that are clean i.e., pass the above two flag checks. These are sources with no foreign >=5 sigma component, and are not cropped; ideally they should have only in-frame white components on the plot, but may also have some very dim contaminants plotted.

In [ ]:
clean_picks = sample_existing(np.where(~foreign & ~cropped)[0], 8)
plotter.plot_grid(clean_picks, suptitle="Clean cutouts (no foreign >=5 sigma source)  |  only the target's own components");

## More information

* **Ellipses are the fitted `Maj`/`Min`/`PA`** (FWHM, beam-convolved), oriented through the cutout WCS.
* Change `RNG`'s seed in the setup cell to inspect a different random sample.
